# Weather Intelligence — sync, embed, retrieve

This notebook runs the whole pipeline against Lakebase and lets you inspect
every stage before deploying the web app.

1. **Harvest** — pull active alerts and narrative forecasts from the National
   Weather Service API and upsert them into `weather_documents`.
2. **Vectorize** — chunk each narrative, embed the chunks with a Databricks
   Model Serving endpoint (`databricks-gte-large-en`, 1024-dim, called over
   REST), and write them into `weather_embeddings` as real pgvector `VECTOR`
   values.
3. **Retrieve** — run cosine-similarity search with pgvector's `<=>` operator
   and read the ranked results.

Everything here calls the same modules the Flask app imports, so what you
validate in this notebook is exactly what the deployed app executes. All
writes go through **pg8000** — there is no Spark JDBC anywhere in the
pipeline, and no compiled C extension either: pg8000 is pure Python, and the
embedding step is a REST call rather than a local model. Both choices avoid
the SIGABRT kernel crash that psycopg2 and sentence-transformers/torch cause
on Databricks serverless compute, including Databricks Free Edition.

## Before you start

Run the scripts in `sql/` against your Lakebase database, in order:
`01_enable_pgvector.sql`, `02_create_weather_documents.sql`,
`03_create_weather_embeddings.sql`. The connection URL comes from the
Databricks secret `database` / `lakebase-url` (see `setup_secrets.py`).

In [ ]:
# Every dependency here is pure Python -- no compiled C extension, no
# torch. That is deliberate: packages like psycopg2 and
# sentence-transformers (which pulls in torch) reliably crash the whole
# kernel with a SIGABRT on Databricks serverless compute, including
# Databricks Free Edition, which is serverless-only. pg8000 is a
# pure-Python Postgres driver, and embeddings come from a Databricks
# Model Serving endpoint called over REST rather than a local model, so
# there is nothing here that can trigger that crash.
%pip install -q --upgrade "databricks-sdk>=0.30.0" "pg8000>=1.31.2" requests pandas

In [ ]:
dbutils.library.restartPython()

## Configuration

Widgets let a scheduled Job override the table names, embedding model and
chunking parameters without editing the notebook. They are written into the
environment *before* `config` is imported, because that module reads its
defaults from environment variables.

In [ ]:
dbutils.widgets.text("documents_table", "weather_documents", "Documents table")
dbutils.widgets.text("embeddings_table", "weather_embeddings", "Embeddings table")
dbutils.widgets.text("embedding_model", "databricks-gte-large-en", "Embedding model")
dbutils.widgets.text("locations", "Chicago, IL;Austin, TX;Denver, CO;Miami, FL;Seattle, WA", "Locations (semicolon separated)")
dbutils.widgets.text("sync_limit", "50", "Max documents per location")
dbutils.widgets.text("chunk_size", "800", "Chunk size (characters)")
dbutils.widgets.text("chunk_overlap", "100", "Chunk overlap (characters)")
dbutils.widgets.text("nws_user_agent", "weather-intelligence-app (your.email@example.com)", "NWS User-Agent contact")

import os
import sys

os.environ["WEATHER_DOCUMENTS_TABLE"] = dbutils.widgets.get("documents_table")
os.environ["WEATHER_EMBEDDINGS_TABLE"] = dbutils.widgets.get("embeddings_table")
os.environ["EMBEDDING_MODEL"] = dbutils.widgets.get("embedding_model")
os.environ["CHUNK_SIZE"] = dbutils.widgets.get("chunk_size")
os.environ["CHUNK_OVERLAP"] = dbutils.widgets.get("chunk_overlap")
os.environ["NWS_USER_AGENT"] = dbutils.widgets.get("nws_user_agent")

LOCATIONS = [part.strip() for part in dbutils.widgets.get("locations").split(";") if part.strip()]
SYNC_LIMIT = int(dbutils.widgets.get("sync_limit"))

# The project modules live one directory up from notebooks/. Put the repo root
# on sys.path so this notebook imports the same code the Flask app runs.
try:
    notebook_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    )
    repo_root = "/Workspace" + os.path.dirname(os.path.dirname(notebook_path))
except Exception:
    repo_root = os.path.dirname(os.getcwd())

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Repo root: {repo_root}")
print(f"Locations: {LOCATIONS}")

In [ ]:
import config
import embedding_pipeline
import lakebase
from weather_client import WeatherClient

print(f"Documents table : {config.WEATHER_DOCUMENTS_TABLE}")
print(f"Embeddings table: {config.WEATHER_EMBEDDINGS_TABLE}")
print(f"Model           : {config.EMBEDDING_MODEL_NAME} ({config.EMBEDDING_DIM}-dim)")
print(f"Chunking        : size={config.CHUNK_SIZE}, overlap={config.CHUNK_OVERLAP}")

## Check the Lakebase connection and schema

`lakebase.ping()` resolves the connection URL from the Databricks secret, opens
a pg8000 connection and reports which instance you are pointed at. If the
tables are missing, run the `sql/` scripts before continuing.

In [ ]:
print(lakebase.ping())

missing = [
    table
    for table in (config.WEATHER_DOCUMENTS_TABLE, config.WEATHER_EMBEDDINGS_TABLE)
    if not lakebase.table_exists(table)
]

if missing:
    raise RuntimeError(
        f"Missing table(s): {missing}. Run the scripts in sql/ against Lakebase first."
    )

print("Schema looks good.")
print(lakebase.run_query(
    "SELECT format_type(a.atttypid, a.atttypmod) AS embedding_column_type "
    "FROM pg_attribute a JOIN pg_class c ON c.oid = a.attrelid "
    "WHERE c.relname = %s AND a.attname = 'embedding'",
    (config.WEATHER_EMBEDDINGS_TABLE,),
))

## Part 1 — Harvest weather narratives

For each location the client resolves coordinates, looks up the NWS forecast
grid cell, then pulls two kinds of free text:

- **Active alerts** — headline, meteorological description, and the
  protective-action instruction, joined into one narrative.
- **Forecast periods** — the `detailedForecast` paragraph for each period of
  the multi-day forecast.

Both are normalized into the same document shape, so one embedding pipeline and
one search endpoint cover both.

In [ ]:
client = WeatherClient()

documents, errors = client.fetch_documents(LOCATIONS, limit=SYNC_LIMIT)

print(f"Harvested {len(documents)} documents from {len(LOCATIONS)} locations")
print(f"  alerts   : {sum(1 for d in documents if d['source_type'] == 'alert')}")
print(f"  forecasts: {sum(1 for d in documents if d['source_type'] == 'forecast')}")
if errors:
    print(f"  skipped  : {errors}")

In [ ]:
# Look at one raw document before it goes anywhere near the database.
if documents:
    sample = documents[0]
    print(f"id           : {sample['id']}")
    print(f"location     : {sample['location']}")
    print(f"source_type  : {sample['source_type']}")
    print(f"headline     : {sample['headline']}")
    print(f"effective_at : {sample['effective_at']}")
    print(f"characters   : {len(sample['narrative_text'])}")
    print()
    print(sample["narrative_text"][:600])
else:
    print("Nothing harvested — try more locations, or a region with active weather.")

In [ ]:
synced = embedding_pipeline.upsert_documents(documents)
print(f"Upserted {synced} documents into {config.WEATHER_DOCUMENTS_TABLE}")
print("Re-running this cell is safe: rows are keyed on a stable id and update in place.")

In [ ]:
import pandas as pd

rows = lakebase.run_query(
    f"""
    SELECT location, source_type, event, severity,
           left(narrative_text, 120) AS narrative_preview,
           effective_at, synced_at
    FROM {config.WEATHER_DOCUMENTS_TABLE}
    ORDER BY synced_at DESC, effective_at DESC NULLS LAST
    LIMIT 20
    """
)
display(pd.DataFrame(rows))

## Part 2 — Chunk and embed

Narratives are split into 800-character windows with 100 characters of overlap.
Most NWS text fits in a single chunk; the window matters for long alert bodies
where the description and the safety instruction together run past the model's
input limit. The overlap keeps a sentence that straddles a boundary intact in at
least one chunk.

Before running the job, look at what the chunker actually produces for the
longest document you harvested.

In [ ]:
longest = lakebase.run_query(
    f"""
    SELECT id, headline, narrative_text, length(narrative_text) AS characters
    FROM {config.WEATHER_DOCUMENTS_TABLE}
    ORDER BY length(narrative_text) DESC
    LIMIT 1
    """
)

if longest:
    doc = longest[0]
    chunks = embedding_pipeline.chunk_text(doc["narrative_text"])
    print(f"{doc['headline']}  ({doc['characters']} characters -> {len(chunks)} chunks)")
    for i, chunk in enumerate(chunks):
        print(f"\n--- chunk {i} ({len(chunk)} chars) ---")
        print(chunk[:220] + ("..." if len(chunk) > 220 else ""))

In [ ]:
# Which documents still need vectors? A document is "pending" when no embedding
# exists for this model AND this exact text, so updated alerts get re-embedded
# while unchanged ones are skipped.
pending = embedding_pipeline.fetch_pending_documents()
print(f"{len(pending)} documents pending embedding")

In [ ]:
result = embedding_pipeline.embed_pending_documents(progress=True)
result

### Confirm the vectors landed as real `VECTOR` values

The embeddings are inserted with an explicit `%s::vector` cast, so there is no
`float8[]` staging column and no follow-up `UPDATE ... ::vector` pass. The query
below reads the stored dimension straight from pgvector.

In [ ]:
display(pd.DataFrame(lakebase.run_query(
    f"""
    SELECT e.id, e.document_id, e.chunk_index, e.model_name,
           vector_dims(e.embedding) AS dims,
           round(vector_norm(e.embedding)::numeric, 4) AS norm,
           left(e.chunk_text, 100) AS chunk_preview
    FROM {config.WEATHER_EMBEDDINGS_TABLE} e
    ORDER BY e.created_at DESC
    LIMIT 10
    """
)))

print(embedding_pipeline.stats())

## Part 3 — Semantic retrieval

The search encodes the query with the same model used at ingestion time and
ranks chunks by cosine distance (`<=>`). Similarity is reported as
`1 - distance`, so 1.0 is an exact match.

Note that these queries share no keywords with the underlying text — "roads may
be slick" is not a phrase the NWS writes — which is the point of vector search
over a `LIKE` filter.

In [ ]:
QUERIES = [
    "flash flood risk this weekend",
    "dangerous heat and humidity",
    "roads may be slick from snow or ice",
    "strong winds that could knock down trees and power lines",
    "clear and pleasant conditions for being outside",
]

for query in QUERIES:
    print(f"\n=== {query} ===")
    for hit in embedding_pipeline.search(query, top_k=3):
        print(f"  {hit['similarity']:.3f}  [{hit['source_type']}] {hit['location']} — {hit['headline']}")
        print(f"         {hit['chunk_text'][:160].replace(chr(10), ' ')}...")

In [ ]:
# Filter retrieval to one source type — useful when you only care about
# safety-critical products rather than routine forecasts.
alerts_only = embedding_pipeline.search(
    "flooding near rivers and creeks", top_k=5, source_type="alert"
)

if alerts_only:
    display(pd.DataFrame(alerts_only)[["similarity", "location", "event", "severity", "chunk_text"]])
else:
    print("No active alerts matched. Alerts only exist when there is live severe weather —")
    print("try a location currently under a watch or warning, or search forecasts instead.")

## Index check

Confirm the HNSW index is present and that the planner is using it. On a small
corpus Postgres may still prefer a sequential scan — that is expected and not a
problem; the index earns its keep as the table grows.

In [ ]:
display(pd.DataFrame(lakebase.run_query(
    "SELECT indexname, indexdef FROM pg_indexes WHERE tablename = %s",
    (config.WEATHER_EMBEDDINGS_TABLE,),
)))

In [ ]:
import time

probe = embedding_pipeline.to_vector_literal(
    embedding_pipeline.embed_texts(["severe thunderstorm with damaging hail"])[0]
)

plan = lakebase.run_query(
    f"""
    EXPLAIN ANALYZE
    SELECT e.id, e.embedding <=> %s::vector AS distance
    FROM {config.WEATHER_EMBEDDINGS_TABLE} e
    ORDER BY e.embedding <=> %s::vector
    LIMIT 5
    """,
    (probe, probe),
)
for line in plan:
    print(list(line.values())[0])

start = time.perf_counter()
embedding_pipeline.search("severe thunderstorm with damaging hail", top_k=5)
print(f"\nEnd-to-end search (encode + query): {(time.perf_counter() - start) * 1000:.1f} ms")

## Next

The data is in Lakebase and retrieval works. Two things you can do now:

- Inspect the tables by hand with `sql/04_verify_schema.sql`.
- Deploy this same repository as a Databricks App and drive the sync and search
  flow from the web UI. See the README for deployment steps.